##Let us import, load and recreate all columns


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

pd.set_option('display.max_colwidth', 120)
sns.set_style('whitegrid')

# Load data
df = pd.read_csv("../data/clean/all_apps_clean.csv")
analyzer = SentimentIntensityAnalyzer()

# VADER functions
def get_vader_score(text):
    if pd.isna(text): return 0.0
    return analyzer.polarity_scores(str(text))['compound']

def vader_label(compound):
    if compound >= 0.05: return 'Positive'
    elif compound <= -0.05: return 'Negative'
    else: return 'Neutral'

def rating_label(rating):
    if rating >= 4: return 'Positive'
    elif rating <= 2: return 'Negative'
    else: return 'Neutral'

# Severity engine
severity_keywords = {
    5: ['locked out', 'account locked', 'account closed', 'money missing',
        'funds missing', 'stolen', 'fraud', 'scam', 'cannot access',
        'lost my money', 'money gone', 'unauthorized'],
    4: ['declined', 'card declined', 'transfer failed', 'verification failed',
        'failed', 'frozen', 'pending', 'verification', "can't send",
        "can't receive", 'not working', 'blocked'],
    3: ['customer service', 'support', 'refund', 'delay', 'late',
        'problem', 'issue', 'error', 'wrong', 'charge'],
    2: ['slow', 'bug', 'glitch', 'annoying', 'confusing', 'difficult'],
    1: ['minor', 'small', 'slight', 'okay', 'fine']
}

def compute_severity(text):
    if pd.isna(text): return 1
    text = str(text).lower()
    for level in [5, 4, 3, 2, 1]:
        for kw in severity_keywords[level]:
            if kw in text:
                return level
    return 1

severity_map = {1:'Low', 2:'Low-Moderate', 3:'Moderate', 4:'High', 5:'Critical'}

# Apply all columns
df['vader_compound']   = df['review_clean'].apply(get_vader_score)
df['vader_sentiment']  = df['vader_compound'].apply(vader_label)
df['rating_sentiment'] = df['rating'].apply(rating_label)
df['severity_score']   = df['review_clean'].apply(compute_severity)
df['severity_label']   = df['severity_score'].map(severity_map)

# Recreate hidden negatives
hidden_neg = df[
    (df['rating_sentiment'] == 'Negative') &
    (df['vader_sentiment']  != 'Negative')
].copy()

print(f"✅ df loaded: {df.shape}")
print(f"✅ Hidden negatives: {len(hidden_neg):,}")

**NOW LET US PREPARE THE TOPIC MODELLING CORPUS**

**We shall focus on moderate to critical hidden negatives only**

**These are the reviews with real businesss pain**

In [ ]:
# Focus on moderate to critical hidden negatives only
# These are the reviews with real business pain
corpus_df = hidden_neg[hidden_neg['severity_score'] >= 3].copy()

# Drop any empty reviews
corpus_df = corpus_df.dropna(subset=['review_clean'])
corpus_df = corpus_df[corpus_df['review_clean'].str.strip() != '']

# Extract the text list BERTopic needs
docs = corpus_df['review_clean'].tolist()

print(f"✅ Corpus size for topic modeling: {len(docs):,} reviews")
print(f"   Severity breakdown:")
print(corpus_df['severity_label'].value_counts())
print(f"\nSample review:")
print(docs[0])

In [ ]:
%pip install bertopic

**NOW LET US RUN BERTOPIC**

In [ ]:
from bertopic import BERTopic

# Initialize and fit BERTopic
topic_model = BERTopic(
    language="english",
    calculate_probabilities=True,
    verbose=True,
    min_topic_size=10
)

topics, probs = topic_model.fit_transform(docs)

# Add topic assignments back to corpus_df
corpus_df = corpus_df.copy()
corpus_df['topic_id'] = topics

# Show topic info
topic_info = topic_model.get_topic_info()
print(f"\n✅ Topics found: {len(topic_info) - 1}")  # -1 excludes outlier topic
display(topic_info.head(15))

**Now let us assigne the names and create the Executive Dashboard**

In [ ]:
# Create a dictionary mapping the topic IDs to meaningful labels
topic_labels = {
    -1: "Miscellaneous/Noise",
    0: "App Performance & Support",
    1: "PayPal Account Disputes",
    2: "Venmo Transaction Friction",
    3: "Chime Banking System Issues",
    4: "AI Support & Bot Frustration",
    5: "Fund Holds & Card Issues",
    6: "Fraud & Scam Protection Failures"
}

corpus_df['topic_name'] = corpus_df['topic_id'].map(topic_labels)

# Show the new breakdown
print("Top Complaint Themes (Hidden Negatives):")
print(corpus_df['topic_name'].value_counts())

Now we look at the **Distributed Pain Complaint themes by app**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# We exclude Topic -1 (Noise) for the final presentation
viz_df = corpus_df[corpus_df['topic_id'] != -1]

plt.figure(figsize=(12, 6))
sns.countplot(data=viz_df, y='topic_name', hue='app_name', palette='viridis')

plt.title("The 'Why' Behind the Miss: Complaint Themes per App\n(Hidden Negatives Only)", fontsize=14, fontweight='bold')
plt.xlabel("Number of Reviews")
plt.ylabel("Complaint Theme")
plt.legend(title="App Name", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("topics_by_app.png", dpi=150, bbox_inches='tight')
plt.show()

Now let us **connect Topics to Severity**

In [ ]:
# Calculate average severity per topic
topic_severity = viz_df.groupby('topic_name')['severity_score'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
topic_severity.plot(kind='barh', color='#8e44ad', edgecolor='black')

plt.title("Average Severity per Complaint Theme", fontsize=14, fontweight='bold')
plt.xlabel("Avg Severity Score (1-5)")
plt.ylabel("Theme")
plt.xlim(1, 5) # Scale to show full range
plt.tight_layout()
plt.savefig("topic_severity.png", dpi=150, bbox_inches='tight')
plt.show()